In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp02/ex_8.py ──
"""
Shared infrastructure for MLFP02 Exercise 8 — FeatureStore + Feature Engineering.

Contains: HDB resale data loading, FeatureStore / ExperimentTracker setup
through kailash-ml, and OLS-from-scratch helpers reused across the four
R10 technique files:

    01_feature_schema.py        — FeatureSchema v1 + validation
    02_point_in_time.py         — Leakage prevention + temporal correctness
    03_rolling_features.py      — FeatureSchema v2 + group_by_dynamic
    04_modeling_with_features.py — Regression + hypothesis tests + Bayes

Technique-specific logic (schema construction, rolling window design,
coefficient interpretation) belongs in the per-technique files. This
module only owns infrastructure and reusable numeric helpers.
"""

from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex8"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_STORE_URL = "sqlite:///mlfp02_ex8_features.db"
FEATURE_TABLE_PREFIX = "kml_feat_"
EXPERIMENT_NAME = "mlfp02_ex8_hdb_features"


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale flats (data.gov.sg)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_resale() -> pl.DataFrame:
    """Load HDB resale transactions with a parsed transaction_date column.

    The raw file stores `month` as "YYYY-MM"; we convert it to a polars
    Date so every downstream technique can sort, filter, and roll on a
    real temporal axis without string parsing.
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    hdb = hdb.with_columns(
        pl.col("month").str.to_date("%Y-%m").alias("transaction_date")
    )
    return hdb


# ════════════════════════════════════════════════════════════════════════
# FEATURE STORE + EXPERIMENT TRACKER — kailash-ml wiring
# ════════════════════════════════════════════════════════════════════════


async def setup_feature_store() -> tuple[Any, Any, Any, bool]:
    """Create (factory, FeatureStore, ExperimentTracker, has_backend).

    Returns ``has_backend=False`` if the infrastructure is unavailable.
    Callers handle the degraded path by running the Polars-only versions
    of each operation.
    """
    try:
        from kailash.infrastructure import StoreFactory
        from kailash_ml import FeatureStore
        from kailash_ml.engines.experiment_tracker import ExperimentTracker

        factory = StoreFactory(FEATURE_STORE_URL)
        fs = FeatureStore(factory, table_prefix=FEATURE_TABLE_PREFIX)
        tracker = ExperimentTracker(factory)
        return factory, fs, tracker, True
    except Exception as exc:  # noqa: BLE001 — degrade gracefully
        print(
            f"  [warn] FeatureStore backend unavailable "
            f"({type(exc).__name__}: {exc})"
        )
        return None, None, None, False


# ════════════════════════════════════════════════════════════════════════
# FEATURE COMPUTATION — v1 (basic property) and v2 (rolling market)
# ════════════════════════════════════════════════════════════════════════


def compute_v1_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute version-1 HDB property features from raw transactions.

    Produces: storey_midpoint, price_per_sqm, remaining_lease_years,
    transaction_id (row index). These are the base features v2 extends.
    """
    return df.with_columns(
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2
        ).alias("storey_midpoint"),
        (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
        (99 - (pl.col("transaction_date").dt.year() - pl.col("lease_commence_date")))
        .cast(pl.Float64)
        .alias("remaining_lease_years"),
    ).with_row_index("transaction_id")


def compute_v2_features(df: pl.DataFrame) -> pl.DataFrame:
    """Compute v2 features = v1 + rolling town-level market context.

    Uses polars ``group_by_dynamic`` on ``transaction_date`` bucketed by
    month, then a 6-month rolling window per town. The first six months
    per town have nulls (warm-up period) — callers must ``drop_nulls``
    before modelling.
    """
    result = compute_v1_features(df).sort("transaction_date")

    town_stats = (
        result.group_by_dynamic("transaction_date", every="1mo", group_by="town")
        .agg(
            pl.col("resale_price").median().alias("monthly_median"),
            pl.col("resale_price").count().alias("monthly_volume"),
        )
        .sort("town", "transaction_date")
    )

    town_stats = town_stats.with_columns(
        pl.col("monthly_median")
        .rolling_mean(window_size=6)
        .over("town")
        .alias("town_median_price"),
        pl.col("monthly_volume")
        .rolling_sum(window_size=6)
        .over("town")
        .alias("town_transaction_volume"),
        (
            (pl.col("monthly_median") - pl.col("monthly_median").shift(6).over("town"))
            / pl.col("monthly_median").shift(6).over("town")
            * 100
        ).alias("town_price_trend"),
    )

    result = result.join(
        town_stats.select(
            "town",
            "transaction_date",
            "town_median_price",
            "town_transaction_volume",
            "town_price_trend",
        ),
        on=["town", "transaction_date"],
        how="left",
    )
    return result


# ════════════════════════════════════════════════════════════════════════
# POINT-IN-TIME RETRIEVAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def as_of(
    df: pl.DataFrame, cutoff: datetime, date_col: str = "transaction_date"
) -> pl.DataFrame:
    """Return rows strictly before ``cutoff`` — the Polars-only PIT path.

    When FeatureStore is unavailable, every technique falls back to this
    helper so the leakage-prevention lesson still runs end-to-end.
    """
    return df.filter(pl.col(date_col) < pl.lit(cutoff.date()))


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH OLS HELPERS — reused across techniques 3 and 4
# ════════════════════════════════════════════════════════════════════════

FEATURE_LIST: list[str] = [
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
    "town_median_price",
    "town_price_trend",
]


def prepare_design_matrix(
    df: pl.DataFrame,
    feature_list: list[str] = FEATURE_LIST,
    target: str = "resale_price",
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Drop nulls, build ``[1, X]`` design matrix, return ``(X, y, names)``."""
    model_data = df.drop_nulls(subset=[*feature_list, target])
    X_raw = model_data.select(feature_list).to_numpy().astype(np.float64)
    y = model_data[target].to_numpy().astype(np.float64)
    X = np.column_stack([np.ones(len(y)), X_raw])
    names = ["intercept", *feature_list]
    return X, y, names


def fit_ols(X: np.ndarray, y: np.ndarray) -> dict[str, Any]:
    """Fit OLS from scratch and return a dict with betas, SEs, t, p, R²."""
    from scipy import stats as sp_stats  # local import — optional at module load

    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_hat = X @ beta
    resid = y - y_hat

    ssr = float(np.sum(resid**2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ssr / sst
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - k)
    rmse = float(np.sqrt(ssr / n))

    sigma_sq = ssr / (n - k)
    xtx_inv = np.linalg.inv(X.T @ X)
    se = np.sqrt(sigma_sq * np.diag(xtx_inv))
    t_stat = beta / se
    p_val = 2.0 * (1.0 - sp_stats.t.cdf(np.abs(t_stat), df=n - k))

    sse = float(np.sum((y_hat - y.mean()) ** 2))
    f_stat = (sse / (k - 1)) / (ssr / (n - k))
    f_p = 1.0 - sp_stats.f.cdf(f_stat, dfn=k - 1, dfd=n - k)

    return {
        "n": n,
        "k": k,
        "beta": beta,
        "se": se,
        "t": t_stat,
        "p": p_val,
        "y_hat": y_hat,
        "resid": resid,
        "r2": float(r2),
        "adj_r2": float(adj_r2),
        "rmse": rmse,
        "f_stat": float(f_stat),
        "f_p": float(f_p),
    }


def normal_normal_posterior(
    beta_hat: float,
    se_hat: float,
    mu_prior: float = 0.0,
    sigma_prior: float = 10_000.0,
) -> dict[str, float]:
    """Normal-Normal conjugate posterior for a single OLS coefficient."""
    prec_prior = 1.0 / sigma_prior**2
    prec_data = 1.0 / se_hat**2
    prec_post = prec_prior + prec_data
    mu_post = (mu_prior * prec_prior + beta_hat * prec_data) / prec_post
    sigma_post = float(np.sqrt(1.0 / prec_post))
    return {
        "mu_post": float(mu_post),
        "sigma_post": sigma_post,
        "ci_low": float(mu_post - 1.96 * sigma_post),
        "ci_high": float(mu_post + 1.96 * sigma_post),
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE SCHEMA BUILDERS — kailash-ml FeatureSchema / FeatureField
# ════════════════════════════════════════════════════════════════════════


def build_schema_v1() -> Any:
    """Return the FeatureSchema v1 definition (basic property features)."""
    from kailash_ml.types import FeatureField, FeatureSchema

    return FeatureSchema(
        name="hdb_property_features",
        features=[
            FeatureField(
                name="floor_area_sqm",
                dtype="float64",
                nullable=False,
                description="Floor area in square metres",
            ),
            FeatureField(
                name="remaining_lease_years",
                dtype="float64",
                nullable=False,
                description="Remaining lease in years",
            ),
            FeatureField(
                name="storey_midpoint",
                dtype="float64",
                nullable=False,
                description="Midpoint of storey range",
            ),
            FeatureField(
                name="price_per_sqm",
                dtype="float64",
                nullable=False,
                description="Transaction price per square metre",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=1,
    )


def build_schema_v2() -> Any:
    """Return FeatureSchema v2 = v1 + three rolling market-context fields."""
    from kailash_ml.types import FeatureField, FeatureSchema

    v1 = build_schema_v1()
    return FeatureSchema(
        name="hdb_property_features",
        features=[
            *v1.features,
            FeatureField(
                name="town_median_price",
                dtype="float64",
                nullable=True,
                description="Median price in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_transaction_volume",
                dtype="int64",
                nullable=True,
                description="Transaction count in town (trailing 6 months)",
            ),
            FeatureField(
                name="town_price_trend",
                dtype="float64",
                nullable=True,
                description="6-month price change % in town",
            ),
        ],
        entity_id_column="transaction_id",
        timestamp_column="transaction_date",
        version=2,
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 8.1: FeatureSchema — Typed Feature Contracts
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Define a FeatureSchema with typed FeatureField entries
#   - Compute base property features from raw HDB resale data
#   - Validate features against the schema contract
#   - Connect to FeatureStore and register/store v1 features
#   - Apply schema-driven feature engineering to Singapore HDB valuation
#
# PREREQUISITES: MLFP02 Exercises 1-7 (Bayesian inference, hypothesis
#   testing, regression, causal inference)
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why typed feature schemas prevent silent failures
#   2. Build — define FeatureSchema v1 and compute features
#   3. Train — register and store features in FeatureStore
#   4. Visualise — feature distributions and correlation structure
#   5. Apply — HDB flat valuation with schema-validated features
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import polars as pl
import plotly.graph_objects as go
from scipy import stats



## THEORY — Why Typed Feature Schemas Prevent Silent Failures


In [ ]:
# A feature schema is a contract between the producer (feature engineer)
# and the consumer (model trainer). Without it, the following failures
# happen silently:
#
#   1. Dtype drift — a feature that was float64 in training becomes
#      string after a data source migration. The model loads, runs, and
#      produces garbage predictions without any error.
#
#   2. Null smuggling — a feature declared "never null" starts getting
#      nulls from a new data partition. The model NaN-propagates through
#      every prediction silently.
#
#   3. Phantom columns — an upstream process renames "price_per_sqm" to
#      "price_sqm". The old column vanishes, the model falls back to
#      defaults, and no one notices until the accuracy report.
#
# FeatureSchema catches ALL THREE at registration time, not at
# prediction time. Think of it as a unit test for your feature contract.
#
# Singapore HDB analogy: HDB publishes standard flat categories
# (3-room, 4-room, 5-room). If a listing arrives as "four-room" instead
# of "4 ROOM", every downstream system that filters by flat_type fails.
# The schema catches this at ingestion, not at report generation.



## TASK 2 — BUILD: Define FeatureSchema v1 and compute property features


In [ ]:
print("\n" + "=" * 70)
print("  Exercise 8.1 — FeatureSchema: Typed Feature Contracts")
print("=" * 70)

# --- 2a. Load HDB resale data ---
# TODO: Call load_hdb_resale() to load the HDB transaction data.
# Hint: load_hdb_resale() returns a Polars DataFrame with a
# transaction_date column already parsed from the month string.
hdb = ____
print(f"\n  Data loaded: {hdb.shape[0]:,} HDB resale transactions")

# --- 2b. Exploratory statistics (Ex 1-2 recap) ---
prices = hdb["resale_price"].to_numpy().astype(np.float64)

# TODO: Compute skewness and kurtosis using scipy.stats.
# Hint: stats.skew(prices) and stats.kurtosis(prices)
skew = ____
kurt = ____

sw_stat, sw_p = stats.shapiro(
    np.random.default_rng(42).choice(prices, size=5000, replace=False)
)

# TODO: Compute MLE estimates for Normal distribution parameters.
# Hint: MLE for Normal: mu = mean, sigma = std with ddof=0
mle_mu = ____
mle_sigma = ____

print(f"\n  Price distribution:")
print(f"    n = {len(prices):,}")
print(f"    Mean: ${mle_mu:,.0f}, Median: ${np.median(prices):,.0f}")
print(f"    Std: ${mle_sigma:,.0f}")
print(
    f"    Skewness: {skew:.3f} "
    f"({'right-skewed' if skew > 0.5 else 'approximately symmetric'})"
)
print(
    f"    Excess kurtosis: {kurt:.3f} "
    f"({'heavy-tailed' if kurt > 1 else 'normal-tailed'})"
)
print(f"    Shapiro-Wilk: W={sw_stat:.4f}, p={sw_p:.6f}")
print(f"\n  MLE (Normal): mu={mle_mu:,.0f}, sigma={mle_sigma:,.0f}")

# --- 2c. Define FeatureSchema v1 ---
# TODO: Call the shared helper to build the v1 schema.
# Hint: build_schema_v1() returns a FeatureSchema with 4 typed fields.
property_schema_v1 = ____

print(f"\n  === FeatureSchema v1 ===")
print(f"  Name: {property_schema_v1.name}, Version: {property_schema_v1.version}")
for f in property_schema_v1.features:
    print(f"    {f.name}: {f.dtype} (nullable={f.nullable}) — {f.description}")

# --- 2d. Compute v1 features ---
# TODO: Call compute_v1_features(hdb) to compute the base features.
# Hint: compute_v1_features adds storey_midpoint, price_per_sqm,
# remaining_lease_years, and transaction_id to the DataFrame.
features_v1 = ____
print(f"\n  Computed v1 features: {features_v1.shape}")

for feat_name in ["price_per_sqm", "storey_midpoint", "remaining_lease_years"]:
    vals = features_v1[feat_name].drop_nulls()
    print(
        f"    {feat_name}: mean={vals.mean():.1f}, "
        f"min={vals.min():.1f}, max={vals.max():.1f}"
    )

# --- 2e. Correlation analysis ---
corr_cols = [
    "resale_price",
    "floor_area_sqm",
    "storey_midpoint",
    "remaining_lease_years",
]

# TODO: Build a correlation matrix from the feature columns.
# Hint: select the corr_cols from features_v1, convert to numpy,
# then use np.corrcoef(data.T) to get the correlation matrix.
corr_data = (
    features_v1.drop_nulls(subset=corr_cols)
    .select(corr_cols)
    .to_numpy()
    .astype(np.float64)
)
corr_matrix = ____

print(f"\n  Correlation matrix:")
print(f"  {'':>20}", end="")
for c in corr_cols:
    print(f"  {c[:12]:>12}", end="")
print()
for i, name in enumerate(corr_cols):
    print(f"  {name[:20]:<20}", end="")
    for j in range(len(corr_cols)):
        print(f"  {corr_matrix[i,j]:>12.3f}", end="")
    print()



### Checkpoint 1


In [ ]:
assert len(prices) > 0, "Task 2: must have price data"
assert "transaction_id" in features_v1.columns, "Task 2: transaction_id missing"
assert "price_per_sqm" in features_v1.columns, "Task 2: price_per_sqm missing"
assert features_v1["price_per_sqm"].min() > 0, "Task 2: price_per_sqm must be positive"
assert corr_matrix.shape == (4, 4), "Task 2: correlation matrix must be 4x4"
print("\n[ok] Checkpoint 1 passed — v1 features computed and validated\n")

# INTERPRETATION: The schema declares 4 features — all non-nullable,
# all float64. This contract means ANY downstream model can trust that
# these columns exist and have no nulls. The storey_midpoint extraction
# from "01 TO 03" → 2.0 is a classic example of feature engineering that
# should be captured in the schema, not rediscovered per model.



## TASK 3 — TRAIN: Register schema and store features in FeatureStore


In [ ]:
# FeatureStore is a versioned, typed feature repository. Registering the
# schema tells the store what columns to expect; storing features
# validates each row against the schema before persisting.

print("\n--- FeatureStore Registration ---")

# TODO: Call setup_feature_store() to connect to the backend.
# Hint: await setup_feature_store() returns
# (factory, fs, tracker, has_backend). has_backend is False if
# the infrastructure is unavailable.
factory, fs, tracker, has_backend = ____

if has_backend:
    try:

        async def store_v1():
            await fs.register_features(property_schema_v1)
            return await fs.store(features_v1, property_schema_v1)

        row_count  = await store_v1()
        print(f"  Stored {row_count:,} v1 feature rows in FeatureStore")
    except Exception as e:
        has_backend = False
        print(f"  [Skipped: FeatureStore ({type(e).__name__}: {e})]")
else:
    print("  [Skipped: FeatureStore backend unavailable]")
    print("  Features remain in-memory as Polars DataFrame — all analysis continues")

# Price by flat type (foreshadows ANOVA concepts from M3)
flat_types = hdb["flat_type"].unique().sort().to_list()
print(f"\n  --- Price by Flat Type ---")
for ft in flat_types:
    subset = hdb.filter(pl.col("flat_type") == ft)["resale_price"]
    if subset.len() > 10:
        print(
            f"    {ft:<12}: n={subset.len():>7,}, "
            f"mean=${subset.mean():>10,.0f}, "
            f"median=${subset.median():>10,.0f}"
        )



### Checkpoint 2


In [ ]:
assert property_schema_v1.version == 1, "Task 3: schema must be version 1"
assert len(property_schema_v1.features) == 4, "Task 3: v1 must have 4 features"
print("\n[ok] Checkpoint 2 passed — schema registered, features stored\n")



## TASK 4 — VISUALISE: Feature distributions and structure


In [ ]:
# Visual proof: the computed features should show plausible
# distributions for Singapore HDB flats. Price_per_sqm should cluster
# around $4,000-8,000/sqm; storey_midpoint should be discrete steps;
# remaining_lease should be 40-99 years.

print("\n--- Feature Distribution Summary ---")
for feat_name in ["price_per_sqm", "storey_midpoint", "remaining_lease_years"]:
    vals = features_v1[feat_name].drop_nulls()
    q25 = vals.quantile(0.25)
    q75 = vals.quantile(0.75)
    print(
        f"  {feat_name:<25} " f"Q1={q25:>8,.1f}  Q3={q75:>8,.1f}  IQR={q75-q25:>8,.1f}"
    )

# TODO: Create a histogram plot with two overlaid distributions.
# Hint: use go.Figure(), add go.Histogram traces for "price_per_sqm"
# and "remaining_lease_years", set barmode="overlay".
fig = go.Figure()
for feat_name in ["price_per_sqm", "remaining_lease_years"]:
    vals = features_v1[feat_name].drop_nulls().to_numpy()
    fig.add_trace(
        go.Histogram(
            x=vals,
            name=feat_name,
            opacity=0.6,
            nbinsx=50,
        )
    )
fig.update_layout(
    title="v1 Feature Distributions — HDB Property Features",
    xaxis_title="Value",
    yaxis_title="Count",
    barmode="overlay",
)
fig.write_html(str(OUTPUT_DIR / "01_feature_distributions.html"))
print(f"\n  Saved: {OUTPUT_DIR / '01_feature_distributions.html'}")

# TODO: Create a correlation heatmap using go.Heatmap.
# Hint: pass z=corr_matrix, x=corr_cols, y=corr_cols,
# colorscale="RdBu_r", zmid=0 for a diverging scale.
fig2 = go.Figure(
    data=go.Heatmap(
        z=____,
        x=corr_cols,
        y=corr_cols,
        colorscale="RdBu_r",
        zmid=0,
        text=np.round(corr_matrix, 3),
        texttemplate="%{text}",
    )
)
fig2.update_layout(title="Feature Correlation Matrix — HDB v1")
fig2.write_html(str(OUTPUT_DIR / "01_correlation_heatmap.html"))
print(f"  Saved: {OUTPUT_DIR / '01_correlation_heatmap.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — feature distributions visualised\n")



## TASK 5 — APPLY: HDB Flat Valuation with Schema-Validated Features


In [ ]:
# Scenario: PropertyGuru Singapore wants to build an automated valuation
# model (AVM) for HDB resale flats. Before any model training, they
# need a reliable feature pipeline with typed schemas.
#
# Without a schema: an upstream change renames "remaining_lease_years"
# to "lease_remaining". The model silently uses a default value.
# Every valuation is wrong. PropertyGuru discovers this 3 weeks later
# when agents report "the system says every flat is worth $350K".
#
# With a schema: the FeatureStore registration rejects the data at
# ingestion. The pipeline fails loudly. PropertyGuru fixes the rename
# in 20 minutes. Zero bad valuations reach agents.
#
# S$ impact: PropertyGuru handles ~15,000 HDB listings/month in
# Singapore. A 3-week silent failure means ~11,250 listings with wrong
# valuations. At an average commission of $8,000 per transaction, even
# a 5% deal-loss rate from bad valuations costs:
#   11,250 * 5% * $8,000 = S$4.5M in lost commission revenue.

print("=== APPLY: HDB Flat Valuation — Schema-Driven Pipeline ===")
print()
print("  Scenario: PropertyGuru automated valuation model (AVM)")
print()
print("  WITHOUT schema:")
print("    - Upstream renames 'remaining_lease_years' to 'lease_remaining'")
print("    - Model silently uses default values for 3 weeks")
print("    - 11,250 listings with wrong valuations")
print("    - S$4.5M lost commission revenue")
print()
print("  WITH FeatureSchema v1:")
print("    - FeatureStore rejects data at ingestion (dtype/name mismatch)")
print("    - Pipeline fails loudly, fixed in 20 minutes")
print("    - Zero bad valuations reach property agents")
print()
print("  Your v1 schema enforces:")
for f in property_schema_v1.features:
    nullable_str = "optional" if f.nullable else "required"
    print(f"    {f.name}: {f.dtype} ({nullable_str})")
print()
print(
    f"  Total features validated: {features_v1.shape[0]:,} rows "
    f"x {len(property_schema_v1.features)} columns"
)



### Checkpoint 4


In [ ]:
print("\n[ok] Checkpoint 4 passed — schema-driven valuation pipeline demonstrated\n")



## REFLECTION


[ok] FeatureSchema: typed fields with dtype, nullable, description
  [ok] FeatureField: individual feature contracts within a schema
  [ok] Feature computation: storey_midpoint, price_per_sqm, remaining_lease
  [ok] FeatureStore registration: schema-validated feature persistence
  [ok] Correlation analysis: identifying multicollinearity early

  KEY INSIGHT: A schema is a unit test for your feature pipeline.
  It catches dtype drift, null smuggling, and phantom columns at
  INGESTION time — not at prediction time when it's too late.

  Next: In 02_point_in_time.py, you'll learn how point-in-time
  retrieval prevents data leakage — the #1 cause of models that
  look great in development and fail catastrophically in production.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

